In [81]:
import numpy as np
from pysr import PySRRegressor

In [82]:
from sympy import Symbol, simplify, expand,symbols, integrate

In [ ]:
import pickle
import torch

# Load the .pkl file
with open('many_well_data_64_dim_with_neg_score_through_score_model_100000.pkl', 'rb') as file:
    data = pickle.load(file)

# Inspect the loaded data
print("Type of data:", type(data))



Type of data: <class 'list'>


In [84]:
data[0].shape

torch.Size([440000, 64])

## Fit PySR model

In [ ]:
def extract_patches(field):
    """Extract patches with zero padding for boundary elements, selecting only center, up, down, left, and right."""
    # Pad the field with zeros
    patches = []
    grid_size = field.shape[0]
    padded_field = np.pad(field, pad_width=1, mode='wrap')

    for i in range(grid_size):
        # for j in range(grid_size):
            if i%2==0:
            # Extract the center and its immediate neighbors
            
             patch = np.array([
                field[ i],
                field[i + 1]   
               ])
             patches.append(patch)
        
    return np.array(patches)

def extract_patches_for_all(configurations):
    """Extract patches for all configurations."""
    all_patches = []
    for config in configurations:
        patches = extract_patches(config)
        all_patches.extend(patches)
    return np.array(all_patches)


num = 1000

patches_all = extract_patches_for_all(((data[0][:100000][::50].cpu().detach().numpy())))

# Flatten 3x3 patches for input to PySR
X = patches_all
y = np.repeat((data[1][:100000][::50]), 32)


# Step 3: Symbolic Regression using PySR
# # Define a custom loss function
# def custom_loss(predictions, targets):
#     """Mean Squared Error between predicted and true total actions."""
#     return np.mean((predictions - targets) ** 2)
# Fit the PySR model
model = PySRRegressor(
    niterations=50,  # Number of search iterations
    binary_operators=["+", "-", "*"],
    unary_operators=["square","cube"],
    #progress=True,
    populations = 30,
    maxsize = 30,
    batching=True,
    batch_size=128,
    verbosity = 1,
    procs = 16
)


In [86]:
X.shape

(64000, 2)

In [ ]:
seed = 4
np.random.seed(seed)
# # Train the model
model.fit(X,y)

/home/ssharma23/.local/lib/python3.10/site-packages/pysr/sr.py:2737: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/home/ssharma23/.local/lib/python3.10/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 1.320e+02
Progress: 11 / 1500 total iterations (0.733%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.188e+00  1.594e+01  y = 1.6911
7           8.688e-01  5.212e-02  y = square((square(x₁) * -0.12589) + -1.1627)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 2.140e+02
Progress: 38 / 1500 total iterations (2.533%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick         score                                           equation  \
	0         0.000000e+00                                          1.6910312   
	1         5.755608e-02                      (x0 * -0.4044296) + 2.1698284   
	2         9.333710e-02             (square(x1) * 0.40579632) - -1.2858778   
	3         2.149121e-04  (square(x1 + -0.02385861) * 0.40551227) - -1.2...   
	4         1.723343e-01  (x0 * -0.40985063) + ((square(x1) * 0.4098567)...   
	5         4.833699e-04  ((square(x1 + -0.029828167) - (x0 - 0.91434276...   
	6         2.466863e-07  ((x1 * -0.024461495) - (square(x1) * -0.409502...   
	7         5.869634e-03  x0 - (((x0 * (1.3973119 - (x0 * -0.12455564)))...   
	8         4.720288e-02  square(x1 * 0.63983685) + (0.9822862 - ((x0 - ...   
	9         7.738494e-04  (square((x1 - 0.03712943) * 0.6384592) - ((x0 ...   
	10        1.392698e-02  ((x0 - cube(x0 - cube(x0))) * 0.0026151373) + ...   
	11        1.619315e-02  (square(x1 * 0.63887626) + 1.7800117) + ((cube...   
	12        2.576933e-02  ((cube((1.017915 - x0) - ((x0 + x0) - cube(x0)...   
	13  >>>>  8.153314e-02  (square((((x0 + x0) - -0.2823898) - cube(x0)) ...   
	14        6.176438e-02  (((square(x1) - ((x0 - 1.6398281) * 0.7762154)...   
	
	        loss  complexity  
	0   1.187850           1  
	1   0.943575           5  
	2   0.859490           6  
	3   0.859120           8  
	4   0.608648          10  
	5   0.608060          12  
	6   0.608060          14  
	7   0.600963          16  
	8   0.573255          17  
	9   0.572369          19  
	10  0.548947          22  
	11  0.531454          24  
	12  0.504757          26  
	13  0.428809          28  
	14  0.378980          30  
]

  - outputs/20250728_153907_HqDwpr/hall_of_fame.csv


In [ ]:
# lowest loss equations
sympy_format[len(model.equations_[0].sympy_format)-1]
sympy_format[len(model.equations_[1].sympy_format)-1]

0.16305627*x0**6 - 0.97833762*x0**4 - 0.0590574515261874*x0**3 + 1.46750643*x0**2 - 0.168017875421438*x0 + 0.406798241226317*x1**2 + 1.4387435276974

## Compute MSE and R2

In [90]:
def mean_normalize(data):
    return (data - np.mean(data))

In [91]:
import pickle
import torch

# Load the .pkl file
with open('many_well_dataset_64_with_neg_logp.pkl', 'rb') as file:
    data_0 = pickle.load(file)
    if isinstance(data_0[0], torch.Tensor):
      variables = data_0[0].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      variables = data_0[0]
    if isinstance(data_0[1], torch.Tensor):
      targets_ = data_0[1].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      targets_ = data_0[1]
features = extract_patches_for_all(variables)
targets = (mean_normalize(targets_))
wn = 32

# Inspect the loaded data
print("Type of data:", type(data_0))

Type of data: <class 'list'>


In [92]:
variables.shape

(20000, 64)

In [ ]:
x0, x1, x2, x3, x4, x5 = symbols('x0 x1 x2 x3 x4 x5')

In [96]:
from sympy import Add, expand

In [ ]:
expr = integrate(expand(model.equations_[0].sympy_format[len(model.equations_[0].sympy_format)-1]),x0)  +  integrate(expand(model.equations_[1].sympy_format[len(model.equations_[1].sympy_format)-1]),x1)

# Reconstruct the expression
eq = expr#sum(new_terms)

(eq)

0.233536429191845*x0**6 - 1.59657133109719*x0**4 + 0.0224275345921642*x0**3 + 2.72873917797582*x0**2 - 0.415515562285433*x0 + 0.43654463*x1**2 + 1.27928037495669

In [99]:
import sympy as sp
from sklearn.metrics import r2_score, mean_squared_error


In [100]:
#calculate mse and r2
symbols_  = eq.free_symbols
# Given expression with fewer variables
x0, x1, x2, x3, x4, x5, x6, x7, x8, x9 = sp.symbols('x0 x1 x2 x3 x4 x5 x6 x7 x8 x9')  # Variables in your potential data
# Step 2: Create a mapping from symbol names to data indices
all_symbols = [x0, x1, x2, x3, x4, x5, x6, x7, x8, x9]  # Known order of variables
symbol_index_map = {symbol: i for i, symbol in enumerate(all_symbols)}
# Step 3: Subset data columns matching variables in the expression
relevant_indices = [symbol_index_map[var] for var in symbols_]
subset_data = features[:, relevant_indices]  # Select relevant columns

#globals().update({str(symbol): sp.symbols(str(symbol)) for symbol in symbols})
numerical_function = sp.lambdify(tuple(symbols_), eq, modules="numpy")
# Step 5: Evaluate the expression for each row
y = numerical_function(*subset_data.T)
y_prd =  mean_normalize(y)
# r2_score

mse = np.round(mean_squared_error(y_pred=mean_normalize(np.sum(np.array(y).reshape(-1,wn),axis=1)),y_true=targets),4)
r2 = np.round(r2_score(y_pred=np.sum(np.array(y_prd).reshape(-1,wn),axis=1),y_true=targets),4)
# with open('extra/results.txt', "a") as file:
#     file.write('multivariate_data_for_multi_mean_unit_cov_with_neg_score_through_nf_model_new.txt' + '\n')
#     file.write(str([(eq),f'mse: {mse}',f'r2: {r2}']) + '\n')
print("mse:",mse,"r2:",r2)

mse: 7.8379 r2: 0.831
